In [1]:
!pip install -q transformers datasets torch scikit-learn pandas numpy psutil accelerate

import os, re, shutil, warnings
import pandas as pd
import numpy as np
import torch
from datasets import load_dataset, Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, EarlyStoppingCallback,
    DataCollatorWithPadding
)
from google.colab import drive

warnings.filterwarnings("ignore")

if not os.path.exists('/content/drive'):
    try:
        drive.mount('/content/drive', force_remount=True)
    except ValueError:
        pass

OUTPUT_DIR = "/content/drive/MyDrive/LIAR_RoBERTa_large_Pro"
os.makedirs(OUTPUT_DIR, exist_ok=True)

try:
    dataset = load_dataset("chengxuphd/liar2")
except Exception:
    dataset = load_dataset("liar")

df_train = pd.DataFrame(dataset['train'])
df_val   = pd.DataFrame(dataset['validation'])
df_test  = pd.DataFrame(dataset['test'])
df = pd.concat([df_train, df_val, df_test], ignore_index=True)

def map_liar_labels_safe(row):
    lbl = row['label']
    if isinstance(lbl, str):
        lbl = lbl.lower()
        if lbl in ['false', 'barely-true', 'pants-fire', 'pants-on-fire']: return 0
        elif lbl in ['true', 'mostly-true', 'half-true']: return 1
        return 0
    if isinstance(lbl, (int, np.integer)):
        if lbl in [0, 4, 5]: return 0
        if lbl in [1, 2, 3]: return 1
        return 0
    return 0

df['binary_label'] = df.apply(map_liar_labels_safe, axis=1)

def create_liar_content(row):
    stmt = str(row.get('statement', '')).strip()
    speaker = str(row.get('speaker', 'Unknown'))
    party = str(row.get('party_affiliation', 'Unknown'))
    context = str(row.get('context', 'Unknown'))
    subject = str(row.get('subject', 'Unknown'))
    meta_info = f"{speaker} ({party}) | {subject} | {context}"
    return stmt + " </s> " + meta_info

df['content'] = df.apply(create_liar_content, axis=1)

def clean_text(s):
    if not isinstance(s, str): return ""
    s = re.sub(r'https?://\S+', ' ', s)
    s = re.sub(r'[^a-zA-Z0-9\s\.\,\!\?\(\)\|\-]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

df['content'] = df['content'].apply(clean_text)
df = df[df['content'].str.len() > 10]

classes = np.unique(df['binary_label'])
class_weights = compute_class_weight('balanced', classes=classes, y=df['binary_label'])
class_weight_dict = {i: float(w) for i, w in zip(classes, class_weights)}

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['binary_label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['binary_label'])

dataset_dict = DatasetDict({
    "train": Dataset.from_pandas(train_df[['content','binary_label']].rename(columns={'binary_label':'label'}).reset_index(drop=True)),
    "validation": Dataset.from_pandas(val_df[['content','binary_label']].rename(columns={'binary_label':'label'}).reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df[['content','binary_label']].rename(columns={'binary_label':'label'}).reset_index(drop=True))
})

MODEL_NAME = "roberta-large"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch["content"], truncation=True, max_length=384, padding=False)

tokenized = dataset_dict.map(tokenize_fn, batched=True, batch_size=1000, remove_columns=['content'])
tokenized = tokenized.rename_column("label", "labels")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.config.id2label = {0: "Fake", 1: "Real"}
model.config.label2id = {"Fake": 0, "Real": 1}

def get_last_checkpoint(output_dir):
    if not os.path.exists(output_dir): return None
    ckpts = [c for c in os.listdir(output_dir) if c.startswith("checkpoint-")]
    if not ckpts: return None
    ckpts_sorted = sorted(ckpts, key=lambda x: int(x.split('-')[-1]), reverse=True)
    return os.path.join(output_dir, ckpts_sorted[0])

last_checkpoint = get_last_checkpoint(OUTPUT_DIR)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weights = [class_weight_dict[0], class_weight_dict[1]]
        w = torch.tensor(weights, dtype=torch.float32, device=model.device)
        loss_fct = torch.nn.CrossEntropyLoss(weight=w)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=3e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    report_to="none"
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train(resume_from_checkpoint=last_checkpoint)
trainer.evaluate(tokenized["test"])

final_path = os.path.join(OUTPUT_DIR, "final_roberta_large_liar")
trainer.save_model(final_path)
tokenizer.save_pretrained(final_path)

for f in os.listdir(OUTPUT_DIR):
    if f.startswith("checkpoint-"):
        shutil.rmtree(os.path.join(OUTPUT_DIR, f), ignore_errors=True)

Mounted at /content/drive


README.md: 0.00B [00:00, ?B/s]

train.csv:   0%|          | 0.00/19.0M [00:00<?, ?B/s]

valid.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [2]:
print(trainer.evaluate(tokenized["test"]))

{'eval_loss': 0.6930841207504272, 'eval_accuracy': 0.6060078363082281, 'eval_f1': 0.4573396086424796, 'eval_precision': 0.3672454976669802, 'eval_recall': 0.6060078363082281, 'eval_runtime': 26.0761, 'eval_samples_per_second': 88.088, 'eval_steps_per_second': 5.522, 'epoch': 3.0}
